# Parte 3: Machine Learning e a História das Personas

Neste capítulo final, utilizamos o algoritmo **K-Means** para encontrar agrupamentos naturais na nossa base de clientes. Através da matemática, revelamos os personagens que compõem o ecossistema do negócio.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
import plotly.express as px

# Carregando os dados preparados no Notebook 02
df_rfm_clean = pd.read_csv('../data/df_rfm_final.csv', index_id='cliente_id')
df_rfm_scaled = pd.read_csv('../data/df_rfm_scaled.csv', index_id='cliente_id')

print("Dados carregados. Pronto para identificar os clusters.")

## 1. O Método do Cotovelo (Elbow Method)

Qual é o número ideal de grupos para esses clientes? Calculamos a **Inércia** (soma das distâncias quadradas dentro dos clusters) para diferentes valores de K. O ponto de inflexão nos indica o equilíbrio ideal entre simplicidade e precisão.

In [ ]:
wcss = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_rfm_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(K_range, wcss, 'bx-')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia (WCSS)')
plt.title('Método do Cotovelo')
plt.grid(True)
plt.show()

## 2. Treinamento do Modelo

Com base no gráfico anterior, escolhemos **K=3**. Agora, deixamos o algoritmo rotular cada um de nossos clientes.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_rfm_clean['Cluster'] = kmeans.fit_predict(df_rfm_scaled)

# Analisando as médias para batizar as personas
df_rfm_clean.groupby('Cluster').agg({
    'Recencia': 'mean',
    'Frequencia': 'mean',
    'Monetario': ['mean', 'count']
}).round(2)

## 3. Interpretação das Personas

Damos nomes aos números para que o time de negócio possa agir:

*   **Campeões (VIPs):** Compram muito, com frequência e estiveram aqui recentemente.
*   **Potenciais:** Clientes novos ou ativos, com bom potencial de crescimento.
*   **Hibernando:** Já gastaram conosco, mas estão há muito tempo sem aparecer. Risco de Churn.

In [ ]:
# Mapeando os nomes e cores
df_rfm_clean['Cluster'] = df_rfm_clean['Cluster'].astype(str)

# Nota: Verifique as médias acima para garantir que o mapeamento de ID -> Nome está correto
persona_map = {
    '0': 'Potenciais',
    '1': 'Hibernando',
    '2': 'Campeões'
}

df_rfm_clean['Persona'] = df_rfm_clean['Cluster'].map(persona_map)

cores_map = {
    'Potenciais': '#f1c40f',
    'Hibernando': '#e74c3c',
    'Campeões': '#2ecc71'
}

df_rfm_clean.head()

## 4. Visualizando a História

Um gráfico 3D nos permite ver claramente como os grupos se separam no espaço de comportamento do consumidor.

In [ ]:
fig = px.scatter_3d(
    df_rfm_clean, 
    x='Recencia', 
    y='Frequencia', 
    z='Monetario',
    color='Persona',
    color_discrete_map=cores_map,
    title='Segmentação RFM: A Visão das Personas',
    opacity=0.8
)
fig.show()

## 5. Aplicação Prática: Matriz de Ação de Marketing

A classificação não deve ser apenas um rótulo; ela deve ser o gatilho para ações automatizadas. Abaixo, uma sugestão de como o time de CRM utilizaria esses dados:

| Persona | Gatilho de Negócio | Ação Recomendada | Canal de Comunicação |
| :--- | :--- | :--- | :--- |
| **Campeões (VIPs)** | Valor Monetário Alto + Recência Baixa | Programa de Fidelidade e Acesso Antecipado. | WhatsApp VIP / Concierge |
| **Potenciais** | Frequência em crescimento + Recência Baixa | Cupom para 2ª/3ª compra e Cross-sell. | Notificação Push / E-mail |
| **Hibernando** | Recência aumentando (>30 dias) | Campanha "Sentimos sua falta" com desconto agressivo. | E-mail Marketing / SMS |

### Exemplo de Automação
Imagine que você precise exportar a lista de clientes **Hibernando** para enviar uma campanha de reativação hoje:

In [ ]:
# Filtrando os clientes em risco (Hibernando) para ação imediata
lista_reativacao = df_rfm_clean[df_rfm_clean['Persona'] == 'Hibernando']

# Exportando para o time de Marketing
lista_reativacao.to_csv('../data/campanha_reativacao_hoje.csv')

print(f"Total de {len(lista_reativacao)} clientes identificados para a campanha de reativação.")